In [2]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys; sys.path.insert(0, str(Path.cwd().parent / "src"))
from prediction.dataset import build_model_table
from prediction.model import fit_ols, coef_table, fit_summary
from prediction.dataset import PREDICTOR_COLS

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
# Build houston and chicago dataframes
hou = build_model_table("houston", refresh=True)
chi = build_model_table("chicago", refresh=True)
tables = {"houston": hou, "chicago": chi} 

Houston city boundary loaded: 1 row(s)
Houston state block groups loaded: 18,626


/home/eprashar_solutions_corelogic_com/crime-idx-2026/src/core/geo_utils.py:27: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  bg['centroid'] = bg.geometry.centroid


Block groups within city: 1,629 / 18,626
Houston crime data: 237,011 rows with valid coords (of 240,696)
Matched to BG: 236,917 | Unmatched: 94
Mapped: 132,286 / 237,011 (55.8%)
Category counts:
crime_category
larceny     67789
vandal      20426
assault     13041
mvt         12355
burglary    11284
robbery      4973
rape         1921
murder        289
fire          208

Unmapped: 104,725 records across 38 codes
BG-level category aggregation: 2,079 BGs (1,600 within city, 479 outside)
Crime Totals:
assault_count      13032.0
murder_count         288.0
rape_count          1921.0
robbery_count       4971.0
burglary_count     11284.0
larceny_count      67775.0
mvt_count          12349.0
vandal_count       20421.0
violent_count      20212.0
property_count     91408.0
cl_total_count    111620.0
dtype: float64
Analysis set: 2,108 BGs
  Inside city:  1,629 (1,600 with data, 29 without)
  Outside city: 479 (all with crime data)
Model loaded: 300,363 total BGs, 2,108 matched to analysis set
Merg

/home/eprashar_solutions_corelogic_com/crime-idx-2026/src/core/geo_utils.py:27: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  bg['centroid'] = bg.geometry.centroid


Chicago crime data: 235,767 rows with valid coords (of 237,207)
Matched to BG: 235,690 | Unmatched: 77
Mapped: 129,757 / 235,767 (55.0%)
Category counts:
crime_category
larceny     58322
vandal      26091
mvt         17164
assault     13640
burglary     6167
robbery      5780
rape         1792
murder        431
fire          370

Unmapped: 106,010 records across 15 codes
BG-level category aggregation: 2,256 BGs (2,164 within city, 92 outside)
Crime Totals:
assault_count      13636.0
murder_count         429.0
rape_count          1792.0
robbery_count       5775.0
burglary_count      6167.0
larceny_count      58295.0
mvt_count          17163.0
vandal_count       26081.0
violent_count      21632.0
property_count     81625.0
cl_total_count    103257.0
dtype: float64
Analysis set: 2,256 BGs
  Inside city:  2,164 (2,164 with data, 0 without)
  Outside city: 92 (all with crime data)
Model loaded: 300,363 total BGs, 2,256 matched to analysis set
Merged comparison set: 2,256 BGs
  Inside city: 

In [4]:
hou.info()

<class 'pandas.DataFrame'>
Index: 1629 entries, 11 to 2102
Data columns (total 46 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   geoid                   1629 non-null   str    
 1   det_pct                 1629 non-null   float64
 2   in_household_pct        1629 non-null   float64
 3   moved1yr_pct            1629 non-null   float64
 4   Division                1629 non-null   float64
 5   city_centers_dist       1629 non-null   float64
 6   pop_est_5mile           1629 non-null   float64
 7   pop_ch_1mile            1629 non-null   float64
 8   vacant_pct              1629 non-null   float64
 9   clip_liens_pct          1629 non-null   float64
 10  clip_foreclosure_pct    1629 non-null   float64
 11  unq_seven_eleven_clips  1629 non-null   float64
 12  unq_gas_station_clips   1629 non-null   float64
 13  unq_liquor_store_clips  1629 non-null   float64
 14  within_city             1629 non-null   bool   
 15  po

#### Separate regressions for Houston and Chicago

In [5]:
# # Run regression on Houston crime
predictors = PREDICTOR_COLS
result, robust, houston_reg = fit_ols(
    df=hou, 
    target="cl_total_logcount"
    )
print(result.summary())
print("="*80)
print(fit_summary(result).to_string(), "\n")
tab = coef_table(result, robust, predictors)
print(tab.to_string())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.182
Model:                            OLS   Adj. R-squared:                  0.176
Method:                 Least Squares   F-statistic:                     29.90
Date:                Thu, 09 Jul 2026   Prob (F-statistic):           2.61e-62
Time:                        21:43:57   Log-Likelihood:                -2413.0
No. Observations:                1629   AIC:                             4852.
Df Residuals:                    1616   BIC:                             4922.
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
const                      3

In [6]:
# Run regression on Chicago crime
predictors = PREDICTOR_COLS
result, robust, chicago_reg = fit_ols(
    df=chi, 
    target="cl_total_logcount"
    )
print(result.summary())
print("="*80)
print(fit_summary(result).to_string(), "\n")
tab = coef_table(result, robust, predictors)
print(tab.to_string())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.270
Model:                            OLS   Adj. R-squared:                  0.266
Method:                 Least Squares   F-statistic:                     66.32
Date:                Thu, 09 Jul 2026   Prob (F-statistic):          1.59e-137
Time:                        21:44:18   Log-Likelihood:                -2358.2
No. Observations:                2164   AIC:                             4742.
Df Residuals:                    2151   BIC:                             4816.
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
const                      3